# Imports

### Installing Required Python Libraries

This cell installs all the additional libraries needed to run the lip-reading pipeline:

- **opencv-python**: For video frame extraction and image processing (e.g., resizing, grayscale conversion).
- **dlib**: Facial landmark detection (used in older pipelines or for comparison).
- **matplotlib**: Plotting and visualizing results (e.g., accuracy curves, histograms).
- **mediapipe**: Modern facial and lip landmark detection (used for extracting the mouth region).
- **g2p-en**: Grapheme-to-phoneme conversion for mapping predicted visemes to phonemes.

These libraries are not included in the default Colab environment, so we explicitly install them here.

In [ ]:
!pip install opencv-python dlib matplotlib
!pip install mediapipe
!pip install g2p-en




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatib

### Preprocessing: Library Imports and Global Configuration

This section prepares the environment for **video preprocessing** and **viseme classification** tasks by importing necessary libraries and setting up key configurations.

---

#### 1. **Core Library Imports**
- **System & File Management**: `os`, `json`, `sys`, `Pathlib`
- **Data Handling**: `pandas`, `numpy`
- **Computer Vision**: `cv2`, `dlib`, `mediapipe`
- **Visualization**: `matplotlib`, `ipywidgets` for interactive controls
- **Performance Tools**: `tqdm` (progress bars), `ThreadPool` (parallel processing), `psutil` (resource monitoring)

#### 2. **Viseme and NLP Tools**
- **NLTK**: Provides phoneme and linguistic resources (e.g., CMU Pronouncing Dictionary, POS tagging, name corpus)
- **g2p-en**: Converts graphemes (letters) to phonemes (speech sounds)
- **inflect**: Converts numbers into words (e.g., `123` → `one hundred twenty-three`)

---

#### 3. **NLTK Resource Initialization**
- Downloads required datasets:
  - `cmudict`: Pronunciation dictionary for phoneme mapping
  - `averaged_perceptron_tagger`: Part-of-speech tagging
  - `names`: Common English names list

#### 4. **Utility Function**
- **`number_to_words(num_str)`**: Converts numeric strings into word equivalents; returns `<unk>` for invalid values.

---

#### 5. **PyTorch and Training Utilities**
- **Model Building**: `torch`, `torch.nn`, `torch.nn.functional`
- **Training Helpers**: `optim`, `DataLoader`, `GradScaler`, `autocast` for mixed-precision training

---

#### 6. **Global Configuration Parameters**
- `device`: Sets training to **GPU (CUDA)** if available; otherwise falls back to **CPU**.
- `SAVE_PATH`: Location where the best-trained model will be saved.
- `NUM_EPOCHS`: Total epochs for training (20 by default).
- `PATIENCE`: Early stopping patience (stop if no improvement after 5 epochs).
- `LEARNING_RATE`: Learning rate for the optimizer (1e-4).
- `WEIGHT_DECAY`: L2 regularization factor (1e-5).
- `BATCH_SIZE`: Number of samples per training batch (32).

---

**Purpose**:  
This cell prepares the entire **software environment** and **model configuration parameters** needed for the lip-reading preprocessing and model training pipeline.


In [ ]:
#Preprocess

import os
import json
import sys
from typing import List, Dict
import cv2
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import dlib
from collections import namedtuple
import mediapipe as mp
from IPython.display import display
import ipywidgets as widgets
import time, psutil
from multiprocessing.pool import ThreadPool
from tqdm import tqdm
from time import time


#Vismes Classification
import nltk
import torch
import inflect
from g2p_en import G2p
from nltk.corpus import cmudict, names

# --- Download required NLTK resources ---
nltk.download('cmudict')
nltk.download('averaged_perceptron_tagger')
nltk.download('names')

# --- Initialize external tools ---
cmu_dict = cmudict.dict()
name_list = set(names.words())
g2p = G2p()
inflect_engine = inflect.engine()

# --- Utility: Convert number to words ---
def number_to_words(num_str):
    try:
        return inflect_engine.number_to_words(int(num_str))
    except:
        return "<unk>"


#Module Build
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# === Global Config ===
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


# === Global Configuration ===
SAVE_PATH = "best_lipreading_model.pt"
NUM_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
BATCH_SIZE=32



[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package names to /root/nltk_data...
[nltk_data]   Package names is already up-to-date!


cpu


### Global Configuration for Preprocessing

This section defines **global constants and tools** required for extracting lip regions and preparing videos for the model.

#### 1. Project Paths
- `PROJECT_ROOT`: Root folder containing datasets, models, and intermediate files.
- `DLIB_SHAPE_PREDICTOR_PATH`: Path to the pre-trained **dlib 68-point facial landmark model**.

#### 2. Directory Management
- `ensure_dir(path)`: Creates directories if they do not exist (used for saving processed outputs).

#### 3. Video Processing Parameters
- **Frame rate (`VIDEO_FPS`)**: Target 25 FPS for uniform frame sampling.
- **Frame size (`FRAME_SIZE`)**: Frames resized to 160×160 before further cropping.
- **Max frames (`MAX_FRAMES`)**: Standardizes all videos to 250 frames for model input.

#### 4. Lip ROI (Region of Interest)
- **ROI Size**: 112×112 pixels.
- **Margins**:
  - `LIP_MARGIN_X = 0.3`: Horizontal margin for stability.
  - `LIP_MARGIN_Y = 0.6`: Vertical margin for stability.
- Defines crop window around mouth landmarks.

#### 5. Landmark Detection
- **Dlib**: Provides basic frontal face detection and 68-point landmarks.
- **MediaPipe FaceMesh**:
  - Modern, high-resolution landmark detector.
  - Detects 468 face landmarks, including detailed lip coordinates (78–88, 308–318).

#### 6. Output Specifications
- **`OUTPUT_SIZE`**: Final lip crop size (112×112).
- **`MOUTH_LANDMARKS`**: Indi**_**


In [ ]:

PROJECT_ROOT = "/content/drive/My Drive/majd"


# Dlib model path
DLIB_SHAPE_PREDICTOR_PATH = os.path.join(PROJECT_ROOT, "dlib_models", "shape_predictor_68_face_landmarks.dat")

print(PROJECT_ROOT)

print(DLIB_SHAPE_PREDICTOR_PATH)

# === Helper to create directories ===
def ensure_dir(directory_path):
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
        print(f" Created directory: {directory_path}")

#extract frames
VIDEO_FPS=25
FRAME_SIZE=(160,160)


detector =dlib.get_frontal_face_detector()
predictor=dlib.shape_predictor(DLIB_SHAPE_PREDICTOR_PATH)


LIP_ROI_SIZE_H = 112
LIP_ROI_SIZE_W = 112
LIP_MARGIN_X = 0.3
LIP_MARGIN_Y = 0.6

MAX_FRAMES=250


OUTPUT_SIZE = (112, 112)
PADDING = 2
MOUTH_LANDMARKS = list(range(78, 88)) + list(range(308, 318))

# --- MediaPipe FaceMesh ---
mp_face_mesh = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.75
)


/content/drive/My Drive/majd
/content/drive/My Drive/majd/dlib_models/shape_predictor_68_face_landmarks.dat


### Loading and Validating Video-Alignment Pairs

This section handles the **loading of video and alignment files** from the project directories (`pretrain` and `main`) and ensures that each video has a corresponding alignment file.

#### 1. `get_files_with_extension(folder, ext)`
- Scans a given folder recursively.
- Collects all files with the specified extension (`.mp4` for videos, `.txt` for alignments).
- Returns a sorted list of absolute file paths.

#### 2. `clean_mismatched_pairs(pairs)`
- Input: List of `VideoPair` namedtuples (video path + alignment path).
- Compares the **base file names** (without extension) to ensure video and alignment belong to the same sample.
- If mismatched, the pair is discarded and a mismatch message is printed.
- Returns a cleaned list and the count of removed mismatches.

#### 3. Dataset Pair Construction
- Defines `VideoPair` namedtuple to link videos and their alignment text files.
- Loads and pairs files for **pretraining dataset** and **main dataset**.
- Cleans mismatches using `clean_mismatched_pairs`.

#### 4. Output Statistics
- Prints number of valid pairs and how many mismatches were removed.
- Shows a **sample pair** (video path and alignment path) to verify data integrity.

---

**Purpose**:  
Ensures that every video has a matching alignment file before preprocessing. This prevents errors during frame extraction and label generation in later stages of the pipeline.


In [ ]:
# --- Load file lists ---
def get_files_with_extension(folder, ext):
    file_paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(ext):
                file_paths.append(os.path.join(root, f))
    return sorted(file_paths)

def clean_mismatched_pairs(pairs):
    """
    Removes pairs where video and alignment base names don't match.

    Args:
        pairs (List[VideoPair]): List of video-alignment namedtuples

    Returns:
        Tuple[List[VideoPair], int]: (cleaned pairs, number of removed pairs)
    """
    cleaned = []
    removed_count = 0

    for pair in pairs:
        video_name = os.path.splitext(os.path.basename(pair.video_path))[0]
        align_name = os.path.splitext(os.path.basename(pair.alignment_path))[0]

        if video_name == align_name:
            cleaned.append(pair)
        else:
            removed_count += 1
            print(f"❌ Mismatch: {video_name} != {align_name}")

    return cleaned, removed_count

PRETRAIN_DIR = os.path.join(PROJECT_ROOT, "pretrain")
MAIN_DIR = os.path.join(PROJECT_ROOT, "main")

pretrain_videos = get_files_with_extension(PRETRAIN_DIR, ".mp4")
pretrain_alignments = get_files_with_extension(PRETRAIN_DIR, ".txt")

main_videos = get_files_with_extension(MAIN_DIR, ".mp4")
main_alignments = get_files_with_extension(MAIN_DIR, ".txt")
# Define a structure
VideoPair = namedtuple("VideoPair", ["video_path", "alignment_path"])

# Build lists of pairs
pretrain_pairs = [
    VideoPair(video_path=v, alignment_path=a)
    for v, a in zip(pretrain_videos, pretrain_alignments)
]

main_pairs = [
    VideoPair(video_path=v, alignment_path=a)
    for v, a in zip(main_videos, main_alignments)
]

# Apply to pretrain and main
pretrain_pairs, pretrain_removed = clean_mismatched_pairs(pretrain_pairs)
main_pairs, main_removed = clean_mismatched_pairs(main_pairs)

print(f"✅ Cleaned pretrain: {len(pretrain_pairs)} pairs, removed: {pretrain_removed}")
print(f"✅ Cleaned main: {len(main_pairs)} pairs, removed: {main_removed}")


#test
print("pretrain :",len(pretrain_pairs))
print("pretrain: ",pretrain_pairs[109].video_path)
print("pretrain: ",pretrain_pairs[109].alignment_path)

print("main :",len(main_pairs))
print("main: ",main_pairs[109].video_path)
print("main: ",main_pairs[109].alignment_path)


❌ Mismatch: 00003 != 00003 (1)
✅ Cleaned pretrain: 96318 pairs, removed: 0
✅ Cleaned main: 48164 pairs, removed: 1
pretrain : 96318
pretrain:  /content/drive/My Drive/majd/pretrain/5536038039829982468/00002.mp4
pretrain:  /content/drive/My Drive/majd/pretrain/5536038039829982468/00002.txt
main : 48164
main:  /content/drive/My Drive/majd/main/5536745420943636139/00071.mp4
main:  /content/drive/My Drive/majd/main/5536745420943636139/00071.txt


### `extract_frames_fixed_length`

This function extracts frames from a video at a **fixed frame rate** and resizes them to a standard size for model input.

#### **Parameters**
- **`video_path` (str)**: Path to the input video file.
- **`target_fps` (int)**: Desired frame rate for sampling frames (default: 25 FPS).
- **`target_size` (tuple)**: Resolution to resize each frame to (default: 122×122).
- **`max_frames` (int)**: Maximum number of frames to extract (default: `MAX_FRAMES = 250`).
- **`grayscale` (bool)**: Whether to convert frames to grayscale (disabled by default).
- **`normalize` (bool)**: Whether to normalize pixel values (disabled by default).

#### **Process**
1. **Video Initialization**
   - Opens the video file using OpenCV’s `VideoCapture`.
   - Reads metadata: original FPS and total frame count.

2. **Timestamp Sampling**
   - Creates an evenly spaced array of timestamps to sample at `target_fps`.
   - Truncates to `max_frames` to ensure consistent output length.

3. **Frame Extraction**
   - For each timestamp:
     - Seeks to the timestamp (in milliseconds).
     - Reads a frame; resizes it to `target_size`.
     - Appends the frame to a list.

4. **Padding**
   - Converts the collected frames to a NumPy array `(T, H, W, 3)`.
   - If the video is shorter than `max_frames`, pads with black frames (zeros) to maintain consistent shape.

#### **Output**
- Returns a NumPy array of shape:


In [ ]:
def extract_frames_fixed_length(
    video_path: str,
    target_fps: int = 25,
    target_size: tuple = (122, 122),
    max_frames: int = MAX_FRAMES,
    grayscale: bool = False,  # Disabled grayscale here
    normalize: bool = False   # Disabled normalization here
):
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"Error: Cannot open video file: {video_path}")
        return None

    original_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / original_fps

    timestamps = np.arange(0, duration, 1 / target_fps)
    timestamps = timestamps[:max_frames]  # Truncate if longer

    frames = []

    for t in timestamps:
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ret, frame = cap.read()
        if not ret:
            break

        if target_size:
            frame = cv2.resize(frame, target_size)

        frames.append(frame)

    cap.release()

    frames_np = np.array(frames)  # Shape: (T, H, W, 3)

    # Pad if shorter than max_frames
    if len(frames_np) < max_frames:
        pad_len = max_frames - len(frames_np)
        pad_shape = (pad_len, *frames_np.shape[1:])
        pad_array = np.zeros(pad_shape, dtype=np.uint8)
        frames_np = np.concatenate([frames_np, pad_array], axis=0)

    return frames_np  # Final shape: (MAX_FRAMES, 122, 122, 3)


### extract_lip_rois_from_frames_fixed

**Purpose**:  
Detects mouth region (ROI) for each frame using facial landmarks, extracts and resizes the mouth area to (112, 112), normalizes pixel values to [0, 1], and ensures output is padded to MAX_FRAMES frames if needed.

**Input**:  
- `frames`: Numpy array of video frames, shape `(MAX_FRAMES, H, W, 3)` (RGB or BGR).  
- `detector`: A face detector object (e.g. dlib.get_frontal_face_detector()).  
- `predictor`: A shape predictor object (e.g. dlib.shape_predictor).  
- `roi_size`: Target size for each mouth ROI.  
- `margin_x`, `margin_y`: Horizontal and vertical margin factors around mouth bounding box.

**Output**:  
- `numpy.ndarray` of shape `(MAX_FRAMES, 112, 112, 1)` – extracted, padded, and normalized mouth ROIs.


In [ ]:
def extract_lip_rois_from_frames_fixed_dlib(
    frames,
    detector,
    predictor,
    roi_size=(112, 112),
    margin_x=0.3,
    margin_y=0.3
):
    mouth_crops = []

    for idx, frame in enumerate(frames):
        try:
            # Convert to grayscale
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            # Detect faces
            faces = detector(gray)
            if len(faces) == 0:
                raise ValueError("No face detected")

            # Predict landmarks
            shape = predictor(gray, faces[0])
            if shape.num_parts != 68:
                raise ValueError("Invalid landmark count")

            # Get lip landmarks
            landmarks = np.array([[p.x, p.y] for p in shape.parts()], dtype=np.int32)
            lip_points = landmarks[48:68]

            # Bounding box with margin
            x_min, y_min = lip_points.min(axis=0)
            x_max, y_max = lip_points.max(axis=0)
            width = x_max - x_min
            height = y_max - y_min
            x_margin = int(width * margin_x)
            y_margin = int(height * margin_y)

            x_min_m = max(0, x_min - x_margin)
            x_max_m = min(gray.shape[1], x_max + x_margin)
            y_min_m = max(0, y_min - y_margin)
            y_max_m = min(gray.shape[0], y_max + y_margin)

            if x_min_m >= x_max_m or y_min_m >= y_max_m:
                raise ValueError("Invalid crop area")

            crop = gray[y_min_m:y_max_m, x_min_m:x_max_m]
            if crop.size == 0:
                raise ValueError("Empty crop")

            # Resize and normalize
            crop_resized = cv2.resize(crop, roi_size, interpolation=cv2.INTER_AREA)
            crop_resized = crop_resized.astype(np.float32) / 255.0
            crop_resized = np.expand_dims(crop_resized, axis=-1)  # Shape: (112, 112, 1)

            mouth_crops.append(crop_resized)

        except Exception as e:
            #print(f"[{idx}] Padding used due to error: {e}")
            mouth_crops.append(np.zeros((*roi_size, 1), dtype=np.float32))

    return np.stack(mouth_crops)  # Shape: (MAX_FRAMES, 112, 112, 1)


### `extract_lip_rois_from_frames_mediapipe`

This function extracts **lip region crops** from a fixed-length sequence of video frames using **MediaPipe FaceMesh**. The resulting crops are preprocessed (grayscale, resized, normalized) and returned as a PyTorch tensor.

---

#### **Parameters**
- **`frames_np` (np.ndarray)**:  
  Input video frames of shape `(MAX_FRAMES, H, W, 3)` in **BGR format** (from OpenCV).

---

#### **Process**

1. **Iterate over Frames**  
   Loops through all `MAX_FRAMES` frames to ensure consistent output length, even if the video is shorter.

2. **Convert to RGB**  
   Each frame is converted from **BGR** to **RGB** for compatibility with MediaPipe.

3. **Facial Landmark Detection**  
   - Uses MediaPipe FaceMesh to detect **468 face landmarks**.
   - Extracts only the **lip landmarks** (indices defined in `MOUTH_LANDMARKS`).

4. **Bounding Box Calculation**  
   - Computes the **min/max x and y** coordinates around the lips.
   - Applies `PADDING` to include margin and prevent cutting lips.

5. **Crop Mouth Region**
   - Extracts the mouth area using the bounding box.
   - Handles failure cases (no landmarks, empty crop) by inserting a **blank frame**.

6. **Preprocess Crop**
   - Converts to **grayscale**.
   - Resizes to `OUTPUT_SIZE` (112×112).
   - Normalizes pixel values to `[0, 1]`.
   - Adds a channel dimension → `(112, 112, 1)`.

7. **Stack and Return**
   - Stacks all processed frames to shape `(MAX_FRAMES, 112, 112, 1)`.
   - Returns as a **PyTorch float32 tensor**.

---

#### **Output**
- A tensor of shape:


In [ ]:

def extract_lip_rois_from_frames_mediapipe(frames_np: np.ndarray) -> torch.Tensor:
    """
    Extracts mouth regions from a fixed-length array of BGR frames using MediaPipe FaceMesh.
    Each region is padded to MAX_FRAMES, converted to grayscale, resized, and normalized.

    Args:
        frames_np (np.ndarray): shape (MAX_FRAMES, H, W, 3) – BGR frames

    Returns:
        torch.Tensor: shape (MAX_FRAMES, 112, 112, 1), dtype float32
    """
    mouth_crops = []

    for idx in range(MAX_FRAMES):
        try:
            frame = frames_np[idx]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = mp_face_mesh.process(rgb)

            if not results.multi_face_landmarks:
                raise ValueError("No landmarks found")

            landmarks = results.multi_face_landmarks[0].landmark
            h, w, _ = frame.shape
            mouth_coords = [(int(landmarks[i].x * w), int(landmarks[i].y * h)) for i in MOUTH_LANDMARKS]

            xs, ys = zip(*mouth_coords)
            x_min = max(min(xs) - PADDING, 0)
            x_max = min(max(xs) + PADDING, w)
            y_min = max(min(ys) - PADDING, 0)
            y_max = min(max(ys) + PADDING, h)

            mouth_crop = frame[y_min:y_max, x_min:x_max]
            if mouth_crop.size == 0:
                raise ValueError("Empty crop")

            gray_crop = cv2.cvtColor(mouth_crop, cv2.COLOR_BGR2GRAY)
            resized = cv2.resize(gray_crop, OUTPUT_SIZE)
            normalized = resized.astype(np.float32) / 255.0
            normalized = np.expand_dims(normalized, axis=-1)  # shape: (112, 112, 1)

            mouth_crops.append(normalized)

        except Exception:
            # Add a blank padded frame if anything goes wrong
            mouth_crops.append(np.zeros((*OUTPUT_SIZE, 1), dtype=np.float32))

    # Stack and return as tensor
    lips_np = np.stack(mouth_crops)  # shape: (MAX_FRAMES, 112, 112, 1)
    return torch.tensor(lips_np, dtype=torch.float32)


# Test

In [ ]:
video_frames = extract_frames_fixed_length(pretrain_pairs[109].video_path)  # (MAX_FRAMES, 122, 122, 1)
lip_tensor = extract_lip_rois_from_frames_mediapipe(video_frames)
print(lip_tensor.shape)

torch.Size([250, 112, 112, 1])


# End To End PreProcesseing Video


### `video_to_preprocced_tensor`

This function defines the **complete preprocessing pipeline** for preparing a video to be used as input for the lip-reading model. It integrates **frame extraction** and **lip ROI cropping** into a single callable function.

---

#### **Parameters**
- **`video_path` (str)**: Path to the input video file to preprocess.
- **Global dependencies**:
  - `MAX_FRAMES`: Maximum frame count (250).
  - `extract_frames_fixed_length`: Function to sample and resize frames.
  - `extract_lip_rois_from_frames_mediapipe`: Function to crop and preprocess lip regions.

---

#### **Process**

1. **Frame Extraction**  
   - Uses `extract_frames_fixed_length()`:
     - Samples frames at 25 FPS.
     - Resizes frames to 122×122 pixels.
     - Pads or truncates to `MAX_FRAMES` (250 frames).

2. **Lip ROI Extraction**  
   - Passes the frames to `extract_lip_rois_from_frames_mediapipe()`:
     - Detects lips using MediaPipe FaceMesh.
     - Crops and normalizes each mouth region.
     - Returns consistent output `(MAX_FRAMES, 112, 112, 1)`.

3. **Tensor Conversion**
   - Converts the NumPy array of preprocessed lip ROIs into a **PyTorch tensor**.
   - Tensor is ready for input to the **3D CNN visual frontend** of the model.

---

#### **Error Handling**
- Returns `None` and logs an error if:
  - Frame extraction fails (e.g., unreadable video).
  - Lip ROI extraction fails (e.g., no detected landmarks).

---

#### **Output**
- A PyTorch tensor of shape:


In [ ]:


def video_to_preprocced_tensor(video_path):
    """
    Full preprocessing pipeline: Takes a video path, extracts frames, crops lip ROIs,
    pads them to MAX_FRAMES frames, and returns a tensor ready for model input.

    Args:
        video_path (str): Path to the input video.
        detector: Dlib face detector object.
        predictor: Dlib facial landmarks predictor object.

    Returns:
        torch.Tensor or None: Tensor of shape (MAX_FRAMES, 112, 112, 1), or None if extraction fails.
    """
    # Step 1: Extract frames from the video
    frames = extract_frames_fixed_length(
        video_path,
        target_fps=25,
        target_size=(122, 122),
        max_frames=MAX_FRAMES,
        grayscale=False,
        normalize=False
    )

    if frames is None:
        print(f"Frame extraction failed for: {video_path}")
        return None

    # Step 2: Extract lip ROIs from the frames
    lips = extract_lip_rois_from_frames_mediapipe(frames)

    if lips is None or len(lips) == 0:
        print(f"Lip ROI extraction failed for: {video_path}")
        return None

    # Step 3: Convert to PyTorch tensor
    lips_tensor = torch.tensor(lips)  # shape: (MAX_FRAMES, 112, 112, 1)

    return lips_tensor


### `video_to_preprocced_npz`

This function is similar to `video_to_preprocced_tensor`, but instead of returning a PyTorch tensor, it outputs a **NumPy array**. This is useful when saving preprocessed data to `.npz` files for efficient storage and batching.

---

#### **Parameters**
- **`video_path` (str)**: Path to the video file that needs preprocessing.

---

#### **Process**

1. **Frame Extraction**  
   - Calls `extract_frames_fixed_length()`:
     - Extracts video frames at 25 FPS.
     - Resizes frames to `122×122`.
     - Pads or truncates to `MAX_FRAMES` frames.

2. **Lip ROI Extraction**  
   - Uses `extract_lip_rois_from_frames_mediapipe()`:
     - Detects facial landmarks.
     - Crops and preprocesses only the **lip region**.
     - Normalizes output and ensures shape `(MAX_FRAMES, 112, 112, 1)`.

3. **Conversion to NumPy Array**  
   - Converts the processed data into a NumPy array with `dtype=float32`.
   - This makes it lightweight and compatible with `.npz` compression.

---

#### **Error Handling**
- Returns `None` if:
  - Frames cannot be extracted.
  - No mouth landmarks are detected.

---

#### **Output**
- NumPy array of shape:


In [ ]:
import numpy as np

def video_to_preprocced_npz(video_path):
    """
    Full preprocessing pipeline: Takes a video path, extracts frames, crops lip ROIs,
    pads them to MAX_FRAMES frames, and returns a NumPy array ready for saving or batching.

    Args:
        video_path (str): Path to the input video.

    Returns:
        np.ndarray or None: Array of shape (MAX_FRAMES, 112, 112, 1), or None if extraction fails.
    """
    # Step 1: Extract frames
    frames = extract_frames_fixed_length(
        video_path,
        target_fps=25,
        target_size=(122, 122),
        max_frames=MAX_FRAMES,
        grayscale=False,
        normalize=False
    )

    if frames is None:
        print(f"❌ Frame extraction failed for: {video_path}")
        return None

    # Step 2: Extract lips
    lips = extract_lip_rois_from_frames_mediapipe(frames)

    if lips is None or len(lips) == 0:
        print(f"❌ Lip ROI extraction failed for: {video_path}")
        return None

    # Step 3: Return as np.ndarray
    return np.array(lips, dtype=np.float32)  # (MAX_FRAMES, 112, 112, 1)


### Interactive Test: Frame Extraction and Lip ROI Visualization

This cell demonstrates and verifies the **preprocessing pipeline** on a single test video by visualizing:
1. Extracted raw frames.
2. Corresponding mouth ROI crops from the lip extraction pipeline.

---

#### **Process**

1. **Select Test Video**
- Uses one of the pretrain dataset videos (`pretrain_pairs[109]`) as a test sample.

2. **Extract Frames**
- Calls `extract_frames_fixed_length()`:
  - Samples frames at **25 FPS**.
  - Resizes to **122×122 pixels**.
  - Converts to **grayscale** for easier visualization.

3. **Extract Lip ROIs**
- Calls `video_to_preprocced_tensor()` to:
  - Detect and crop the mouth region using **MediaPipe FaceMesh**.
  - Normalize to **112×112** resolution.
  - Return preprocessed mouth crops as a tensor.

4. **Interactive Visualization**
- Uses **Matplotlib** to display:
  - The original extracted frame (left).
  - The corresponding cropped mouth ROI (right).
- A **slider widget** (`ipywidgets.IntSlider`) allows interactive navigation through frames.

---

#### **Purpose**
- Validate that the preprocessing steps work correctly:
  - Frame sampling produces consistent output.
  - Mouth landmarks are detected properly.
  - Crops align with lip movements throughout the video.

- Serves as a quick quality check before batch preprocessing and training.


In [ ]:


# --- Your test video ---
video_path = pretrain_pairs[109].video_path

frames = extract_frames_fixed_length(
    video_path,
    target_fps=25,
    target_size=(122, 122),
    max_frames=MAX_FRAMES,
    grayscale=True,     # grayscale ON for visualization
    normalize=False
)

# Normalize and squeeze frames for display
if frames is None:
    print("Frame extraction failed.")
else:
    frames_display = frames.astype("float32") / 255.0  # Shape: (MAX_FRAMES, 122, 122, 1)

    # Step 2: Extract lips using your pipeline
    lips_tensor = video_to_preprocced_tensor(video_path)

    if lips_tensor is None:
        print("Lip ROI extraction failed.")
    else:
        lips_display = lips_tensor.numpy()  # Shape: (MAX_FRAMES, 112, 112, 1)

        # Step 3: Interactive display
        def show_frame_and_mouth(idx):
            fig, axs = plt.subplots(1, 2, figsize=(10, 5))

            # Original frame
            frame = frames_display[idx].squeeze()
            axs[0].imshow(frame, cmap='gray')
            axs[0].set_title(f"Original Frame {idx}")
            axs[0].axis("off")

            # Mouth ROI
            mouth = lips_display[idx].squeeze()
            axs[1].imshow(mouth, cmap='gray')
            axs[1].set_title("Mouth ROI")
            axs[1].axis("off")

            plt.tight_layout()
            plt.show()

        slider = widgets.IntSlider(min=0, max=frames_display.shape[0] - 1, step=1, value=0, description='Frame')
        widgets.interact(show_frame_and_mouth, idx=slider)


<ipython-input-9-3362548836>:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lips_tensor = torch.tensor(lips)  # shape: (MAX_FRAMES, 112, 112, 1)


interactive(children=(IntSlider(value=0, description='Frame', max=249), Output()), _dom_classes=('widget-inter…

# Visemes & Special Tokens

### Viseme and Token Configuration

This cell defines the **core token vocabulary** and mapping logic for the lip-reading system, including visemes, phonemes, and special tokens required for training and decoding.

---

#### **1. Viseme Tokens**
- `VISEME_TOKENS`:  
  Contains **41 viseme classes** (vocal mouth shapes) commonly used in speech-to-lip mapping, derived from **CMU Pronouncing Dictionary** conventions.

Examples:
- Vowels: `AA`, `AE`, `AH`, `AO`, ...
- Consonants: `B`, `CH`, `D`, `DH`, `F`, ...
- Special silent token: `sil`
- Unknown token: `<unk>`

---

#### **2. Viseme → Phoneme Mapping**
- `viseme_to_phonemes`:
  - Each viseme maps to a **set of phonemes** (with stress variants like `AA0`, `AA1`).
  - Example:  
    `AE` → `["AE", "AE0", "AE1", "AE2"]`

---

#### **3. Phoneme → Viseme Mapping**
- Builds an **inverted dictionary**:
  - Allows quick lookup of which viseme corresponds to a given phoneme.
  - Useful during **phoneme-to-viseme conversion** after G2P (grapheme-to-phoneme) processing.

---

#### **4. Special Tokens**
- `SPECIAL_TOKENS`:
  - Reserved tokens for **sequence control**:
    - `<sos>`: Start of sequence  
    - `<eos>`: End of sequence  
    - `<sow>` / `<eow>`: Word boundaries  
    - `<space>`: Explicit space token  
    - `<sil>`: Silence  
    - `<pad>`: Padding for sequence alignment

---

#### **5. Combined Vocabulary**
- `ALL_TOKENS`:
  - Concatenates **special tokens** and **viseme tokens** into a single list.
- `TOKEN_TO_INDEX`:
  - Maps each token to a **unique integer index** for training and inference.

---

#### **Purpose**
This configuration enables:
- Consistent tokenization of viseme sequences.
- Accurate conversion between phoneme and viseme representations.
- Proper handling of sequence boundaries during Transformer training and prediction.


In [ ]:
# Complete list of viseme classes (based on CMU and common mapping)
VISEME_TOKENS  = [
    "AA", "AE", "AH", "AO", "AW", "AY",
    "B", "CH", "D", "DH", "EH", "ER",
    "EY", "F", "G", "HH", "IH", "IY",
    "JH", "K", "L", "M", "N", "NG",
    "OW", "OY", "P", "R", "S", "SH",
    "T", "TH", "UH", "UW", "V", "W",
    "Y", "Z", "ZH", "sil","<unk>"
]


# Viseme to Phoneme map (expanded, hardcoded)
viseme_to_phonemes = {
    "AA": ["AA", "AA0", "AA1", "AA2"],"AE": ["AE", "AE0", "AE1", "AE2"],"AH": ["AH", "AH0", "AH1", "AH2"],"AO": ["AO", "AO0", "AO1", "AO2"],
    "AW": ["AW", "AW0", "AW1", "AW2"],"AY": ["AY", "AY0", "AY1", "AY2"],"B": ["B"],"CH": ["CH"],"D": ["D"],"DH": ["DH"],
    "EH": ["EH", "EH0", "EH1", "EH2"],"ER": ["ER", "ER0", "ER1", "ER2"],"EY": ["EY", "EY0", "EY1", "EY2"],"F": ["F"],"G": ["G"],
    "HH": ["HH"],"IH": ["IH", "IH0", "IH1", "IH2"],"IY": ["IY", "IY0", "IY1", "IY2"],"JH": ["JH"],"K": ["K"],"L": ["L"],"M": ["M"],
    "N": ["N"],"NG": ["NG"],"OW": ["OW", "OW0", "OW1", "OW2"],"OY": ["OY", "OY0", "OY1", "OY2"],"P": ["P"],"R": ["R"],"S": ["S"],
    "SH": ["SH"],"T": ["T"],"TH": ["TH"],"UH": ["UH", "UH0", "UH1", "UH2"],"UW": ["UW", "UW0", "UW1", "UW2"],"V": ["V"],
    "W": ["W"],"Y": ["Y"],"Z": ["Z"],"ZH": ["ZH"],"sil": ["sil"],"unk":["unk"]
}


# Invert the dictionary to get phoneme → viseme
phoneme_to_viseme = {}
for viseme, phonemes in viseme_to_phonemes.items():
    for phoneme in phonemes:
        phoneme_to_viseme[phoneme] = viseme

SPECIAL_TOKENS = ["<sos>", "<eos>", "<sow>", "<eow>", "<space>", "<sil>","<pad>"]



# Combine and create a token → index dictionary
ALL_TOKENS = SPECIAL_TOKENS + VISEME_TOKENS
TOKEN_TO_INDEX = {tok: idx for idx, tok in enumerate(ALL_TOKENS)}


### Alignment to Viseme Sequence Conversion

This section defines functions to **parse alignment files** and convert spoken words into **flat viseme token sequences** (ready for training).

---

#### **1. `extract_words_from_alignment(path)`**
- Reads `.txt` alignment files.
- Skips metadata lines starting with `"Text:"`.
- Extracts the **word sequence** in spoken order.
- Returns a list of words, e.g., `["THESE", "DAYS", "WHEN", "YOU'RE", ...]`.

---

#### **2. `words_to_visemes_with_g2p(word_list, phoneme_to_viseme_map)`**
- Converts words → phonemes → visemes using **two-step fallback**:
  1. **CMU Pronouncing Dictionary** (preferred).
  2. **G2P (grapheme-to-phoneme)** model if word not in CMU.
- Handles:
  - **Digits** (e.g., `"5"` → `"five"`).
  - Strips stress markers (`AH0` → `AH`).
- Outputs viseme sequences for each word, e.g.,  
  `"THESE"` → `["DH", "IY", "Z"]`.

---

#### **3. `format_viseme_sequence_with_tokens_flat(words, word_to_visemes)`**
- Combines all viseme sequences into **one flat sequence**.
- Adds **special tokens**:
  - `<sos>`: Start of sequence
  - `<sow>` / `<eow>`: Word boundaries
  - `<space>`: Space between words
  - `<eos>`: End of sequence
- Example output:  
  `['<sos>', '<sow>', 'DH', 'IY', 'Z', '<eow>', '<space>', '<sow>', 'D', 'EY', 'Z', '<eow>', '<eos>']`

---

#### **4. `pad_viseme_sequence_to_length(viseme_sequence, target_length=MAX_FRAMES)`**
- Pads the sequence to a **fixed length** (e.g., 250 tokens).
- Uses `<sil>` (silence) as padding token.
- Ensures all training sequences have identical lengths for batching.

---

#### **Purpose**
- Converts alignment files into **tokenized viseme sequences** with consistent length.
- These sequences align with video frame tensors for **Transformer training** and **evaluation**.


In [ ]:
def extract_words_from_alignment(path):
    """
    Extracts the sequence of spoken words from an alignment .txt file, skipping headers like 'Text:'.

    Args:
        path (str): Path to the alignment file.

    Returns:
        List[str]: List of words in spoken order.
    """
    words = []
    try:
        with open(path, 'r') as f:
            lines = f.readlines()

        for line in lines:
            line = line.strip()
            if line.startswith("Text:"):
                continue  # Skip "Text:" line
            parts = line.split()
            if len(parts) >= 4 and parts[0] != "WORD":
                words.append(parts[0])

    except Exception as e:
        print(f"Error reading alignment file {path}: {e}")

    return words



# Example phoneme_to_viseme mapping (must be defined earlier in your code)
# phoneme_to_viseme = {"AA": "AA", "AE": "AE", ..., "ZH": "ZH", "sil": "sil"}

from nltk.corpus import names
import re

# Prepare name list
try:
    all_known_names = set(names.words())
except LookupError:
    import nltk
    nltk.download('names')
    from nltk.corpus import names
    all_known_names = set(names.words())

def words_to_visemes_with_g2p(word_list, phoneme_to_viseme_map):
    """
    Converts a list of words to viseme sequences using CMU dict and g2p fallback.

    Args:
        word_list (List[str]): List of words.
        phoneme_to_viseme_map (Dict[str, str]): Map from phoneme to viseme.

    Returns:
        List[List[str]]: List of viseme sequences.
    """
    result = []

    for word in word_list:
        word_lower = word.lower()

        # Convert digit strings to English words (e.g., "5" → "five")
        if word_lower.isdigit():
            word_lower = number_to_words(word_lower)

        viseme_seq = []

        if word_lower in cmu_dict:
            # Use CMU dict if available
            phoneme_seq = cmu_dict[word_lower][0]
        else:
            # Use g2p fallback
            phoneme_seq = g2p(word_lower)
            phoneme_seq = [p for p in phoneme_seq if p not in [" ", "", ".", "<unk>"]]

        # Convert phonemes to visemes (strip stress like AH0 → AH)
        for phoneme in phoneme_seq:
            base_phoneme = phoneme.strip("012")
            viseme = phoneme_to_viseme_map.get(base_phoneme, "<unk>")
            viseme_seq.append(viseme)

        result.append(viseme_seq)

    return result


def format_viseme_sequence_with_tokens_flat(words, word_to_visemes):
    """
    Converts a list of words into a flat viseme sequence with special tokens.

    Args:
        words (List[str]): Spoken word sequence.
        word_to_visemes (Callable): Function that maps a word to a list of viseme tokens.

    Returns:
        List[str]: Flat sequence like: ['<sos>', '<sow>', 'DH', 'IY', 'Z', '<eow>', '<space>', ..., '<eos>']
    """
    sequence = ["<sos>"]

    for word in words:
        visemes = word_to_visemes(word)
        if visemes:
            sequence.append("<sow>")
            sequence.extend(visemes)  # Add each viseme individually
            sequence.append("<eow>")
            sequence.append("<space>")

    if sequence[-1] == "<space>":
        sequence.pop()

    sequence.append("<eos>")
    return sequence


def pad_viseme_sequence_to_length(viseme_sequence, target_length=MAX_FRAMES, pad_token="<sil>"):
    """
    Pads a viseme sequence to a fixed length with <sil> tokens.

    Args:
        viseme_sequence (List[str]): List of viseme tokens with special markers.
        target_length (int): Desired fixed length for the sequence (default = MAX_FRAMES).
        pad_token (str): Token used for padding (default = "<sil>").

    Returns:
        List[str]: Padded viseme sequence of exactly target_length tokens.
    """
    current_len = len(viseme_sequence)

    if current_len >= target_length:
        return viseme_sequence[:target_length]

    padding = [pad_token] * (target_length - current_len)
    return viseme_sequence + padding






# End To End Preparing Alignment For a video

### Convert Alignment File → Padded Viseme Tensor

This function transforms an **alignment text file** into a **fixed-length viseme token tensor** used for training.

---

#### **Function: `alignment_to_viseme_tensor(path)`**

**Input:**
- `path` (str): Path to the alignment `.txt` file.

**Process:**
1. **Extract words**:
   - Uses `extract_words_from_alignment()` to read and clean the word sequence from the file.

2. **Convert words → visemes**:
   - Uses `words_to_visemes_with_g2p()` to map each word to a list of visemes (via CMU dictionary or G2P fallback).

3. **Format with special tokens**:
   - Adds `<sos>`, `<sow>`, `<eow>`, `<space>`, and `<eos>` using `format_viseme_sequence_with_tokens_flat()`.

4. **Pad to fixed length**:
   - Pads with `<sil>` tokens up to `MAX_FRAMES` using `pad_viseme_sequence_to_length()`.

5. **Map to indices**:
   - Converts each viseme token into an integer ID using `TOKEN_TO_INDEX`.

6. **Convert to tensor**:
   - Returns a `torch.LongTensor` of shape `(MAX_FRAMES,)`.

---

#### **Output:**
- A **padded tensor** of token indices ready to be used as:
  - **Model target (decoder input)**
  - **Ground-truth label** for loss computation

**Example:**
```python
tensor = alignment_to_viseme_tensor("/path/to/alignment.txt")
print(tensor.shape)   # torch.Size([250])


In [ ]:

def alignment_to_viseme_tensor(path):
    """
    End-to-end: converts alignment to a padded tensor of token indices.

    Args:
        path (str): Path to alignment text file.

    Returns:
        torch.Tensor: Long tensor of shape (MAX_FRAMES,) ready for training.
    """
    words = extract_words_from_alignment(path)
    viseme_seqs = words_to_visemes_with_g2p(words, phoneme_to_viseme)

    # Flatten with tokens
    flat_sequence = format_viseme_sequence_with_tokens_flat(words, lambda w: viseme_seqs[words.index(w)])

    # Pad to MAX_FRAMES
    padded_sequence = pad_viseme_sequence_to_length(flat_sequence)

    # Convert to token indices
    token_ids = [TOKEN_TO_INDEX.get(tok, TOKEN_TO_INDEX["<sil>"]) for tok in padded_sequence]

    # Convert to tensor
    return torch.tensor(token_ids, dtype=torch.long)


### Convert Alignment File → Viseme Token Index Array (NumPy)

This function transforms an **alignment text file** into a **fixed-length viseme token index array** for saving into `.npz` files or batching for training.

---

#### **Function: `alignment_to_viseme_npz(alignment_path)`**

**Input:**
- `alignment_path` (str): Path to the alignment `.txt` file.

**Process:**
1. **Extract words**  
   - Reads spoken words from the alignment file using `extract_words_from_alignment()`.

2. **Convert words → visemes**  
   - Uses `words_to_visemes_with_g2p()` (CMU dict or G2P fallback) to convert each word into viseme sequences.

3. **Flatten and add special tokens**  
   - Adds `<sos>`, `<sow>`, `<eow>`, `<space>`, `<eos>` to form a single viseme sequence using `format_viseme_sequence_with_tokens_flat()`.

4. **Pad to fixed length**  
   - Pads with `<sil>` tokens up to `MAX_FRAMES` using `pad_viseme_sequence_to_length()`.

5. **Convert tokens to indices**  
   - Maps each viseme token to an integer index from `TOKEN_TO_INDEX`.

6. **Return as NumPy array**  
   - Returns a padded `np.ndarray` of shape `(MAX_FRAMES,)` with `dtype=int64`.

---

#### **Output:**
- A **fixed-length NumPy array** of viseme token indices, suitable for:
  - Saving into `.npz` files
  - Loading directly into PyTorch tensors during training

**Example:**
```python
viseme_array = alignment_to_viseme_npz("/path/to/alignment.txt")
print(viseme_array.shape)   # (250,)


In [ ]:
import numpy as np

def alignment_to_viseme_npz(alignment_path):
    """
    Converts an alignment file to a viseme token index NumPy array (padded).

    Args:
        alignment_path (str): Path to alignment text file.

    Returns:
        np.ndarray: Array of shape (MAX_FRAMES,) containing token indices.
    """
    # Step 1: Extract words from alignment
    words = extract_words_from_alignment(alignment_path)

    # Step 2: Map words to viseme sequences using G2P + viseme map
    viseme_seqs = words_to_visemes_with_g2p(words, phoneme_to_viseme)

    # Step 3: Flatten with special tokens
    flat_sequence = format_viseme_sequence_with_tokens_flat(
        words, lambda w: viseme_seqs[words.index(w)]
    )

    # Step 4: Pad to MAX_FRAMES
    padded_sequence = pad_viseme_sequence_to_length(flat_sequence)

    # Step 5: Convert tokens to indices
    token_ids = [TOKEN_TO_INDEX.get(tok, TOKEN_TO_INDEX["<sil>"]) for tok in padded_sequence]

    # Step 6: Return as np.ndarray
    return np.array(token_ids, dtype=np.int64)


# Testing


In [ ]:
# Step 1: Build index-to-token map
index_to_token = {v: k for k, v in TOKEN_TO_INDEX.items()}

# Step 2: Run full pipeline
viseme_tensor = alignment_to_viseme_tensor(pretrain_alignments[0])

# Step 3: Print results
print("Tensor shape:", viseme_tensor.shape)

token_ids = viseme_tensor.tolist()
print("Token IDs:", token_ids)

decoded_tokens = [index_to_token[i] for i in token_ids]
print("Decoded Tokens:", decoded_tokens)


Tensor shape: torch.Size([250])
Token IDs: [0, 2, 16, 24, 44, 3, 4, 2, 15, 19, 44, 3, 4, 2, 42, 17, 29, 3, 4, 2, 43, 39, 34, 3, 4, 2, 26, 39, 26, 23, 30, 3, 4, 2, 14, 23, 33, 35, 3, 4, 2, 8, 37, 3, 4, 2, 22, 31, 28, 3, 4, 2, 16, 9, 3, 4, 2, 37, 34, 9, 15, 23, 36, 9, 29, 9, 27, 3, 4, 2, 14, 23, 33, 3, 4, 2, 33, 8, 29, 3, 4, 2, 10, 20, 9, 29, 3, 4, 2, 35, 37, 19, 44, 3, 4, 2, 7, 29, 3, 4, 2, 16, 9, 3, 4, 2, 36, 17, 27, 20, 3, 4, 2, 23, 29, 3, 4, 2, 20, 8, 41, 11, 34, 3, 4, 2, 9, 41, 3, 4, 2, 9, 3, 4, 2, 13, 19, 26, 23, 30, 3, 4, 2, 37, 34, 19, 3, 4, 2, 9, 29, 15, 3, 4, 2, 9, 3, 4, 2, 13, 8, 21, 3, 4, 2, 9, 41, 3, 4, 2, 20, 34, 31, 44, 9, 29, 3, 4, 2, 9, 41, 9, 29, 3, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
Decoded Tokens: ['<sos>', '<sow>', 'DH', 'IY', 'Z', '<eow>', '<space>', '<sow>', 'D', 'EY', 'Z', '<eow>', '<space>', '<sow>', 'W

# with Saving

### Function: `process_and_save_pair`

This function processes a single video-alignment pair, converts them into tensors, and saves them for training.

---

#### **Purpose**
- Automates **preprocessing** for both video frames and alignment text.
- Produces **two tensor files** per pair:
  1. Preprocessed video tensor (lip ROIs).
  2. Padded viseme token tensor.
- Saves tensors with a **unique index** in the specified directory.

---

#### **Parameters**
- `pair (Tuple[str, str])`  
  A tuple containing paths to the video and its corresponding alignment file.

- `save_dir (str)`  
  Directory where the processed tensors will be saved.

- `index (int)`  
  Unique integer used to name saved files (`processed_tensor_{index}.pt`, `vismes_tensor_{index}.pt`).

---

#### **Process**
1. **Ensure save directory exists**  
   Creates `save_dir` if it doesn’t exist.

2. **Process video**  
   - Calls `video_to_preprocced_tensor(video_path)`:
     - Extracts frames.
     - Crops lip ROIs via MediaPipe.
     - Pads and normalizes frames.
     - Returns tensor of shape `(MAX_FRAMES, 112, 112, 1)`.

3. **Save video tensor**  
   - Saves with filename: `processed_tensor_{index}.pt`.

4. **Process alignment**  
   - Calls `alignment_to_viseme_tensor(alignment_path)`:
     - Converts words to phonemes → visemes.
     - Adds `<sow>`, `<eow>`, `<space>` tokens.
     - Pads to `MAX_FRAMES`.
     - Returns token index tensor `(MAX_FRAMES,)`.

5. **Save viseme tensor**  
   - Saves with filename: `vismes_tensor_{index}.pt`.

6. **Return paths**  
   Returns tuple: `(processed_tensor_path, viseme_tensor_path)`.

---

#### **Example Usage**
```python
save_dir = "/content/processed_data"
video_path = pretrain_pairs[0].video_path
alignment_path = pretrain_pairs[0].alignment_path

processed_path, viseme_path = process_and_save_pair(
    (video_path, alignment_path), save_dir, index=0
)

print("Video tensor saved to:", processed_path)
print("Viseme tensor saved to:", viseme_path)


In [ ]:
def process_and_save_pair(pair, save_dir, index):
    """
    Given a pair (video_path, alignment_path), processes both, saves tensors,
    and returns paths to the saved files.

    Args:
        pair (Tuple[str, str]): (video_path, alignment_path)
        save_dir (str): Directory to save the output tensors.
        index (int): Index used to name the saved files uniquely.

    Returns:
        Tuple[str, str]: (processed_tensor_path, visemes_tensor_path)
    """
    video_path, alignment_path = pair

    # Create save_dir if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)

    # Process video → tensor
    video_tensor = video_to_preprocced_tensor(video_path)
    if video_tensor is None:
        print(f"Video processing failed: {video_path}")
        return None, None

    processed_tensor_path = os.path.join(save_dir, f"processed_tensor_{index}.pt")
    torch.save(video_tensor, processed_tensor_path)

    # Process alignment → viseme tensor
    viseme_tensor = alignment_to_viseme_tensor(alignment_path)
    if viseme_tensor is None:
        print(f"Alignment processing failed: {alignment_path}")
        return processed_tensor_path, None

    viseme_tensor_path = os.path.join(save_dir, f"vismes_tensor_{index}.pt")
    torch.save(viseme_tensor, viseme_tensor_path)

    return processed_tensor_path, viseme_tensor_path

### Function: `process_pair_as_npz`

This function processes a video-alignment pair and returns **NumPy arrays** instead of saving them as `.pt` files.  
It is optimized for **batch processing and later saving into compressed `.npz` datasets**.

---

#### **Purpose**
- Convert raw video and alignment into **model-ready NumPy arrays**.
- Keep the data in memory for flexible storage (e.g., batch `.npz` files).
- Avoid immediate saving to disk (used for building large dataset batches).

---

#### **Parameters**
- `pair (Tuple[str, str])`  
  A tuple containing:
  1. `video_path`: Path to `.mp4` video.
  2. `alignment_path`: Path to `.txt` alignment file.

---

#### **Returns**
- `Tuple[np.ndarray, np.ndarray]`  
  - **video_np**: `(MAX_FRAMES, 112, 112, 1)`  
    Normalized lip ROI frames.  
  - **viseme_np**: `(MAX_FRAMES,)`  
    Token indices representing viseme sequence with special tokens.  

- `(None, None)` if processing fails (video or alignment error).

---

#### **Processing Steps**
1. **Video Preprocessing**
   - Calls `video_to_preprocced_npz(video_path)`:
     - Extracts frames at fixed FPS.
     - Crops lips using MediaPipe FaceMesh.
     - Converts to grayscale and pads to `MAX_FRAMES`.
     - Returns NumPy array `(MAX_FRAMES, 112, 112, 1)`.

2. **Alignment Preprocessing**
   - Calls `alignment_to_viseme_npz(alignment_path)`:
     - Extracts words → phonemes → visemes.
     - Adds `<sos>`, `<sow>`, `<eow>`, `<space>`, `<eos>` tokens.
     - Pads to `MAX_FRAMES`.
     - Returns NumPy array `(MAX_FRAMES,)`.

3. **Error Handling**
   - If either processing step fails, returns `(None, None)`.

---

#### **Example Usage**
```python
video_np, viseme_np = process_pair_as_npz(pretrain_pairs[0])

if video_np is not None and viseme_np is not None:
    print("Video shape:", video_np.shape)
    print("Viseme shape:", viseme_np.shape)


In [ ]:
def process_pair_as_npz(pair):
    """
    Processes a (video_path, alignment_path) pair and returns both as NumPy arrays.

    Args:
        pair (Tuple[str, str]): (video_path, alignment_path)

    Returns:
        Tuple[np.ndarray, np.ndarray] or (None, None):
            - video_np: shape (MAX_FRAMES, 112, 112, 1)
            - viseme_np: shape (MAX_FRAMES,)
    """
    video_path, alignment_path = pair

    # Process video
    video_np = video_to_preprocced_npz(video_path)
    if video_np is None:
        print(f"❌ Video processing failed: {video_path}")
        return None, None

    # Process alignment
    try:
        viseme_np = alignment_to_viseme_npz(alignment_path)
    except Exception as e:
        print(f"❌ Viseme processing failed: {alignment_path} | {e}")
        return None, None

    return video_np, viseme_np


# Without saveing

### `process_pair_without_saving(pair)`

---

#### **Purpose**
- Process a single `(video_path, alignment_path)` pair.
- Return **video tensor** and **viseme tensor** directly in memory (no saving to disk).

---

#### **Inputs**
- `pair` *(Tuple[str, str])*:  
  A tuple containing:
  - `video_path`: Path to the `.mp4` video file.
  - `alignment_path`: Path to the corresponding `.txt` alignment file.

---

#### **Process**
1. **Video Processing**
   - Calls `video_to_preprocced_tensor(video_path)`:
     - Extracts frames (fixed length).
     - Crops **lip ROI** using MediaPipe.
     - Pads to `MAX_FRAMES`.
     - Returns tensor shape `(MAX_FRAMES, 112, 112, 1)`.

2. **Alignment Processing**
   - Calls `alignment_to_viseme_tensor(alignment_path)`:
     - Extracts words from alignment file.
     - Converts words → phonemes → visemes.
     - Adds special tokens (`<sow>`, `<eow>`, `<sos>`, `<eos>`).
     - Pads to `MAX_FRAMES`.
     - Returns tensor shape `(MAX_FRAMES,)`.

3. **Validation**
   - Returns `(None, None)` if either step fails.

---

#### **Returns**
- **Success**: `(video_tensor, viseme_tensor)`
- **Failure**: `(None, None)`

---

#### **Usage Example**
```python
video_tensor, viseme_tensor = process_pair_without_saving(pretrain_pairs[0])
print(video_tensor.shape, viseme_tensor.shape)


In [ ]:
def process_pair_without_saving(pair):
    """
    Given a (video_path, alignment_path) pair, processes both and returns the tensors
    without saving them to disk.

    Args:
        pair (Tuple[str, str]): (video_path, alignment_path)

    Returns:
        Tuple[torch.Tensor, torch.Tensor] or (None, None) on failure
    """
    video_path, alignment_path = pair

    # Process video → tensor
    video_tensor = video_to_preprocced_tensor(video_path)
    if video_tensor is None:
        print(f"[ERROR] Video processing failed: {video_path}")
        return None, None

    # Process alignment → viseme tensor
    viseme_tensor = alignment_to_viseme_tensor(alignment_path)
    if viseme_tensor is None:
        print(f"[ERROR] Alignment processing failed: {alignment_path}")
        return None, None

    return video_tensor, viseme_tensor


#taking a list with the size of BATCH_NUMBER pair paths and saves them in the /content/pretrain_data_batches_ready_to_load

In [ ]:
def save_npz_batch_from_pairs_in_memory(pairs, batch_index, save_root="/content/pretrain_data_batches_ready_to_load"):
    import os
    import numpy as np
    from time import time

    os.makedirs(save_root, exist_ok=True)

    video_batch, viseme_batch = [], []
    skipped = 0
    start_time = time()

    for video_path, alignment_path in pairs:
        video_np, viseme_np = process_pair_as_npz((video_path, alignment_path))
        if video_np is None or viseme_np is None:
            skipped += 1
            continue

        # ✅ Convert to float16 for smaller size
        video_batch.append(video_np.astype(np.float16))
        viseme_batch.append(viseme_np.astype(np.int64))

    if not video_batch:
        print(f"⚠️ No valid samples in batch {batch_index}")
        return None

    try:
        videos_np = np.stack(video_batch)    # (B, T, 112, 112, 1)
        visemes_np = np.stack(viseme_batch)  # (B, T)
        save_path = os.path.join(save_root, f"{batch_index:04d}.npz")
        np.savez_compressed(save_path, videos=videos_np, visemes=visemes_np)
        print(f"✅ Saved compressed batch {batch_index:04d} | {len(video_batch)} samples | Skipped: {skipped} | Time: {time() - start_time:.2f}s")
        return save_path
    except Exception as e:
        print(f"❌ Failed to save batch {batch_index:04d} | Error: {e}")
        return None


### `compress_npy_file(input_path, compressed_path=None)`

---

#### **Purpose**
- Converts a `.npy` file into a **compressed `.npz` file** to save storage space.
- Removes the original `.npy` file after compression.

---

#### **Parameters**
- `input_path` (str):
  - Path to the `.npy` file to be compressed.
- `compressed_path` (str, optional):
  - Desired output path for the compressed `.npz` file.
  - Defaults to `<original_filename>_compressed.npz` in the same folder.

---

#### **Returns**
- Path to the compressed `.npz` file (str).

---

#### **Workflow**
1. Validate file extension (`.npy` required).
2. Load the `.npy` file into memory.
3. Save compressed version with `np.savez_compressed()`.
4. Delete the original `.npy` file to reclaim space.

---

#### **Example Usage**
```python
compressed_path = compress_npy_file("/content/data/0001_videos.npy")
print(compressed_path)  # /content/data/0001_videos_compressed.npz


In [ ]:
import os
import numpy as np
from time import time

def compress_npy_file(input_path: str, compressed_path: str = None) -> str:
    if not input_path.endswith('.npy'):
        raise ValueError("Input path must end with .npy")

    if compressed_path is None:
        dirpath = os.path.dirname(input_path)
        filename = os.path.basename(input_path)[:-4]
        compressed_path = os.path.join(dirpath, f"{filename}_compressed.npz")

    arr = np.load(input_path, allow_pickle=False)
    np.savez_compressed(compressed_path, arr=arr)
    os.remove(input_path)  # Remove uncompressed .npy to save space
    return compressed_path


def save_npz_batch_from_pairs_in_memory_unit8(pairs, batch_index, save_root="/content/pretrain_data_batches_ready_to_load"):
    os.makedirs(save_root, exist_ok=True)

    video_batch, viseme_batch = [], []
    skipped = 0
    start_time = time()

    for video_path, alignment_path in pairs:
        video_np, viseme_np = process_pair_as_npz((video_path, alignment_path))
        if video_np is None or viseme_np is None:
            skipped += 1
            continue

        video_uint8 = (video_np * 255).clip(0, 255).astype(np.uint8)
        video_batch.append(video_uint8)
        viseme_batch.append(viseme_np.astype(np.int64))

    if not video_batch:
        print(f"⚠️ No valid samples in batch {batch_index}")
        return None

    try:
        videos_np = np.stack(video_batch)    # (B, T, 112, 112, 1)
        visemes_np = np.stack(viseme_batch)  # (B, T)

        # Save each array as separate uncompressed .npy
        video_npy_path = os.path.join(save_root, f"{batch_index:04d}_videos.npy")
        viseme_npy_path = os.path.join(save_root, f"{batch_index:04d}_visemes.npy")

        np.save(video_npy_path, videos_np)
        np.save(viseme_npy_path, visemes_np)

        # Compress each file separately
        compressed_video_path = compress_npy_file(video_npy_path)
        compressed_viseme_path = compress_npy_file(viseme_npy_path)

        print(f"✅ Saved compressed batch {batch_index:04d} | {len(video_batch)} samples | Skipped: {skipped} | Time: {time() - start_time:.2f}s")
        print(f"📦 Video: {compressed_video_path}")
        print(f"📦 Viseme: {compressed_viseme_path}")
        return compressed_video_path, compressed_viseme_path

    except Exception as e:
        print(f"❌ Failed to save batch {batch_index:04d} | Error: {e}")
        return None


### Batch Processing and Saving Pretrain Data

This cell processes all video–alignment pairs from the `pretrain_pairs` list and saves them as **compressed `.npz` batches** for efficient loading during training.

**Key points:**
- **Batch Size:** 32 videos per `.npz` file (each `.npz` contains `videos` and `visemes` arrays).
- **Directory:** Saved in `/content/pretrain_data_batches_ready_to_load`.
- **Loop Logic:**  
  - Iterates through dataset in sequential 32-video chunks.  
  - Stops automatically if `pretrain_pairs` ends before `MAX_BATCH_INDEX`.  
- **Compression:** Videos are stored as `uint8` (0–255) and visemes as `int64` to reduce storage size.
- **Progress Output:** Prints skipped videos, processed batch count, and total runtime.

**Usage:**
- Adjust `MAX_BATCH_INDEX` depending on dataset size (e.g., `len(pretrain_pairs)//32`).
- Ensure `pretrain_pairs` is fully populated and paths are valid.
- Outputs two files per batch:  
  - `####_videos_compressed.npz`  
  - `####_visemes_compressed.npz`


In [ ]:
BATCH_SIZE = 32
SAVE_ROOT = "/content/pretrain_data_batches_ready_to_load"
MAX_BATCH_INDEX = 2500

# Create directory if it doesn't exist
os.makedirs(SAVE_ROOT, exist_ok=True)

from time import time

start_time = time()
total_pairs_saved = 0
total_skipped = 0

for batch_index in range(1, MAX_BATCH_INDEX):  # from batch 1 to 2499
    start = batch_index * BATCH_SIZE
    end = start + BATCH_SIZE
    batch_pairs = [(p.video_path, p.alignment_path) for p in pretrain_pairs[start:end]]

    if not batch_pairs:
        print(f"⚠️ No data found for batch {batch_index}")
        continue

    print(f"🔄 Starting batch {batch_index:04d} with {len(batch_pairs)} pairs")

    result = save_npz_batch_from_pairs_in_memory_unit8(
        pairs=batch_pairs,
        batch_index=batch_index,
        save_root=SAVE_ROOT
    )

    if result is not None:
        total_pairs_saved += len(batch_pairs)
    else:
        total_skipped += 1

print("\n=== ✅ Summary ===")
print(f"Total pairs processed (estimate): {total_pairs_saved}")
print(f"Batches skipped (no valid data): {total_skipped}")
print(f"Total time: {time() - start_time:.2f} seconds")


🔄 Starting batch 0001 with 32 pairs
✅ Saved compressed batch 0001 | 32 samples | Skipped: 0 | Time: 99.28s
📦 Video: /content/pretrain_data_batches_ready_to_load/0001_videos_compressed.npz
📦 Viseme: /content/pretrain_data_batches_ready_to_load/0001_visemes_compressed.npz
🔄 Starting batch 0002 with 32 pairs
✅ Saved compressed batch 0002 | 32 samples | Skipped: 0 | Time: 92.58s
📦 Video: /content/pretrain_data_batches_ready_to_load/0002_videos_compressed.npz
📦 Viseme: /content/pretrain_data_batches_ready_to_load/0002_visemes_compressed.npz
🔄 Starting batch 0003 with 32 pairs
✅ Saved compressed batch 0003 | 32 samples | Skipped: 0 | Time: 83.09s
📦 Video: /content/pretrain_data_batches_ready_to_load/0003_videos_compressed.npz
📦 Viseme: /content/pretrain_data_batches_ready_to_load/0003_visemes_compressed.npz
🔄 Starting batch 0004 with 32 pairs
✅ Saved compressed batch 0004 | 32 samples | Skipped: 0 | Time: 83.11s
📦 Video: /content/pretrain_data_batches_ready_to_load/0004_videos_compressed.npz

#test

In [ ]:
import os
import numpy as np

# Path to the batch
batch_path_video = "/content/pretrain_data_batches_ready_to_load/0001_videos_compressed.npz"
batch_path_visemes = "/content/pretrain_data_batches_ready_to_load/0001_visemes_compressed.npz"

# Load the .npz file (use key 'arr' for single-array .npz)
data1 = np.load(batch_path_video)
data2 = np.load(batch_path_visemes)

video_array = data1["arr"]
viseme_array = data2["arr"]

# Check shape
print("✅ Shape check for batch 0001:")
print("videos shape :", video_array.shape)    # Expected: (32, MAX_FRAMES, 112, 112, 1)
print("visemes shape:", viseme_array.shape)   # Expected: (32, MAX_FRAMES)

# Check size on disk
size_video_mb = os.path.getsize(batch_path_video) / (1024 * 1024)
size_viseme_mb = os.path.getsize(batch_path_visemes) / (1024 * 1024)

print(f"📦 Video file size on disk: {size_video_mb:.2f} MB")
print(f"📦 Viseme file size on disk: {size_viseme_mb:.2f} MB")


✅ Shape check for batch 0001:
videos shape : (32, 250, 112, 112, 1)
visemes shape: (32, 250)
📦 Video file size on disk: 19.08 MB
📦 Viseme file size on disk: 0.00 MB


In [1]:
print((len(pretrain_pairs)//32) * 21)

NameError: name 'pretrain_pairs' is not defined

### 🧩 Full Model: Lip Reading Transformer

This class encapsulates the entire end-to-end lip reading architecture, combining:

1. **Visual Frontend**: A 3D CNN that extracts spatiotemporal features from grayscale mouth-region video frames.
2. **Transformer Encoder**: Models long-range temporal dependencies across visual features.
3. **Transformer Decoder**: Generates structured viseme token sequences (`<sos>`, visemes, `<eow>`, `<space>`, `<eos>`) from the encoded features.

---

#### 🔢 Input

- **Video tensor**: `(B, T, 112, 112, 1)`  
  A batch of videos with `T` grayscale frames (e.g. 250).
- **Target token tensor** *(only during training)*: `(B, L)`  
  Each sequence contains token indices including special tokens like `<sos>` and `<eos>`.

---

#### 🔁 Training Flow

1. Input video is passed through the **3D CNN** → outputs `(B, T', 512)`
2. Output is processed by the **Transformer Encoder** → shape remains `(B, T', 512)`
3. Target tokens are passed to the **Transformer Decoder**, which generates logits for each output position.
4. Final output: `(B, L, V)` where `V` is the viseme vocabulary size.

---

#### 🔎 Inference Flow

- Starts from `<sos>` token.
- Decodes tokens autoregressively one at a time using previously predicted tokens and encoder memory.
- Stops when `<eos>` is predicted or `max_seq_len` is reached.

---

#### 🧠 Token Sequence Structure (Ground Truth)

Your ground-truth viseme sequences should be **fully structured** during training, including:

- `<sos>` at the beginning  
- A sequence of tokens for each word:
  - Begins with `<sow>`
  - Ends with `<eow>`
  - Separated by `<space>` if another word follows  
- `<eos>` at the end

**Example for sentence**: *"hi majd"*  
Token sequence:  
`<sos> <sow> HH AY <eow> <space> <sow> M AH J D <eow> <eos>`

This structure allows the decoder to learn when a word ends and whether to continue with another.

---

#### 📌 Notes

- The model performs many-to-many sequence prediction (not frame-to-token aligned).
- It supports variable-length viseme sequences via padding and masking.
- `<sow>`, `<eow>`, and `<space>` are predicted by the decoder — they must be included in training labels.
